# C · Tune current oxygen-concern rules

**Purpose:** compare deterministic rules for detecting a declared event in measured SpO₂. This notebook does not train a blackout classifier, calibrate MAX30102 optics, or establish a safe underwater oxygen limit. Its selected configuration is marked **research-only / not deployable**.

Unlike Notebook A's prediction of a future relative drop, this notebook detects an event already occurring. By default the reference event is **source SpO₂ ≤90% for at least five elapsed seconds**. The number 90, the severity endpoints 95/85, and all timing settings below are **explicit numerical benchmark choices**, not clinical recommendations. Reference labels come from the released oximeter readings themselves. Good agreement therefore establishes event-response behavior, not independent measurement accuracy or medical validity.

**Run on Kaggle:** import this notebook, choose Python and CPU / no accelerator, enable Internet, then Run All. No API key or other notebook is required. It automatically verifies and downloads about 9 MB from [dry apnea (80 people)](https://zenodo.org/records/21701010), [HAPP (20 people)](https://doi.org/10.6084/m9.figshare.29947679), and [UW controlled hypoxemia (6 people)](https://github.com/ubicomplab/oximetry-phone-cam-data). These are dry/source-domain recordings, not swimming or blackout examples. This notebook uses oxygen; missing HR does **not** make oxygen ineligible.

**Offline:** attach the unpacked original files through Add Input and set `ALLOW_DOWNLOAD=False` below. Original filenames/checksums must be preserved; the loader searches recursively under `/kaggle/input`. A/C use identical source files. Outputs go to `/kaggle/working/oxygen_concern`; download that output folder after Save Version / Run All. `QUICK=True` tests a smaller participant subset and a reduced parameter grid; its results are only a smoke test.

**Session baseline:** a pre-immersion / pre-intervention baseline is not inferred from an arbitrary first minute. The optional drop branch stays off unless you provide verified baseline annotations. This makes the default runnable comparison a level/trend/persistence benchmark; it does not pretend to fit the full session-baseline policy from absent metadata.

In [ ]:
import importlib.util
required = {'numpy':'numpy', 'pandas':'pandas', 'openpyxl':'openpyxl',
            'requests':'requests', 'matplotlib':'matplotlib', 'IPython':'ipython'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ImportError('Missing packages: '+', '.join(missing)+
                     '. Install only these in a separate %pip install cell, then restart and Run All.')
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from collections import deque
import hashlib, itertools, json, math, os, platform, re, time
import numpy as np
import pandas as pd
import openpyxl, requests
import matplotlib.pyplot as plt
from IPython.display import display

def env_flag(name, default):
    return os.getenv(name, str(default)).lower() in {'1', 'true', 'yes'}

DATA_ROOT = Path(os.getenv('CONCERN_DATA_ROOT', '/kaggle/input'))
DEFAULT_OUT = '/kaggle/working/oxygen_concern' if Path('/kaggle').exists() else 'oxygen_concern_output'
OUT = Path(os.getenv('CONCERN_OUTPUT_DIR', DEFAULT_OUT)); OUT.mkdir(parents=True, exist_ok=True)
FIGURES = OUT / 'figures'; FIGURES.mkdir(exist_ok=True)
CACHE = Path(os.getenv('CONCERN_CACHE', '/kaggle/temp/model_a_sources' if Path('/kaggle').exists()
                       else str(OUT.parent / 'model_a_source_cache')))
ALLOW_DOWNLOAD = env_flag('CONCERN_DOWNLOAD', True)
QUICK = env_flag('CONCERN_QUICK', False)
SOURCE_HOLDOUT = env_flag('CONCERN_SOURCE_HOLDOUT', True)
SOURCES = ['dry_apnea', 'happ', 'uw']
SEED = 2026

# Numerical experiments only; none is a validated wearable alarm setting.
EVENT_SPO2 = 90.0
EVENT_DURATION_SECONDS = 5   # Six consecutive 1-Hz observations span five seconds.
LEVEL_START, LEVEL_FULL = 95.0, 85.0
ENTER_SCORE, EXIT_SCORE = 0.50, 0.25
MAX_UNMATCHED_PER_HOUR = 2.0
MAX_NON_EVENT_WARNING_FRACTION = 0.10
VERIFIED_BASELINE_CSV = os.getenv('CONCERN_BASELINE_CSV', '')
BASELINE_DROP_START, BASELINE_DROP_FULL = 2.0, 6.0
assert LEVEL_START > EVENT_SPO2 > LEVEL_FULL
assert 0 <= EXIT_SCORE < ENTER_SCORE <= 1
assert EVENT_DURATION_SECONDS >= 1
assert MAX_UNMATCHED_PER_HOUR >= 0 and 0 <= MAX_NON_EVENT_WARNING_FRACTION <= 1
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
print('Research-only numerical endpoint:', EVENT_SPO2, '% for', EVENT_DURATION_SECONDS, 'elapsed seconds')
print('Quick:', QUICK, '| Sources:', SOURCES, '| Outputs:', OUT.resolve())
print('Libraries:', {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__})

## Verified source acquisition and participant split

Preserve the original pairing and timestamps; do not join independent people by row number. HAPP includes source-side forward filling without per-sample flags. UW uses Nellcor `SpO2 2`; its other fields are not interchangeable. The same source-qualified participant split as Notebook A is used, before any labels or rule selection. Original files are checksum pinned. No download is silently substituted when an archive changes.

In [ ]:
HAPP_FILES = [
    ("0609b.csv", 57303830, "0f107f209da5523d8f2c8b9407e54e11"),
    ("0610b.csv", 57303833, "62e41f18b4e20d7c641bba5544c51207"),
    ("0613b.csv", 57303836, "ffb57c13160aec154e392eb3ee2f8d3e"),
    ("0620a.csv", 57303839, "06dbd2713689f3ae3cc406e119c09095"),
    ("0620b.csv", 57303842, "20ad187a4f87abc0a631a20fef8f20f0"),
    ("0623a.csv", 57303845, "0392bfadce426ec6274a8a18699845fa"),
    ("0624a.csv", 57303848, "8a7437600bccd94641428007ae953b46"),
    ("0624b.csv", 57303851, "cdad3a47e0444ffb08091abf97d89392"),
    ("0626a.csv", 57303854, "14b19ecfdc45118a299ffa65119c7aa1"),
    ("0626b.csv", 57303857, "3406d6a0d4f8f3f07e8a7fbecac81ca2"),
    ("0627a.csv", 57303860, "8ba2cb0a6d738bee4fbaa2081038506f"),
    ("0714a.csv", 57303863, "d304ef294a2f4026b1874250a7a037b2"),
    ("0714b.csv", 57303866, "b114142022139784a7607877641d8d82"),
    ("0715a.csv", 57303869, "d25c537a49504549921f5098bec4747e"),
    ("0717b.csv", 57303872, "fd5de165a346d9b7b49f0f82cfac04f6"),
    ("0721b.csv", 57303875, "ddede47bc85bdf0e22cddc8c85cead96"),
    ("0722a.csv", 57303878, "9172d814e5173d179397353dbe046843"),
    ("0722b.csv", 57303881, "ace5f34ee7a9458863d82ab56f552f71"),
    ("0723b.csv", 57303884, "e0f48e773c817274635662331090cf82"),
    ("0728b.csv", 57303887, "36cf7e074f53d879f9fbc1b2a3d6d88f"),
]
UW_FILES = [
    ("100001.csv", "1a82a222775583e7dfa8df9e5061cda897fdde83"),
    ("100002.csv", "d84cc7d0a12435d69b67e47b1f493d5cf0aefb3c"),
    ("100003.csv", "9110ad6ebccbd1f3417fadf81a4e071ea4d7426c"),
    ("100004.csv", "136cc180ff34084ca791521c18ad2183ae10188f"),
    ("100005.csv", "5c63857e43e258fc590619f04b99792411e5d9e1"),
    ("100006.csv", "c6afaf83a83095c0780428f60769c66d3993615e"),
]
UW_COMMIT = "c483ae8fe4cd32b0b9ce412d07299b05279fc58f"
manifest = [("dry_apnea", "Supp database.xlsx",
             "https://zenodo.org/records/21701010/files/Supp%20database.xlsx?download=1",
             "md5", "79fc9abbd6e3e04c52d17df3247751ec")]
manifest += [("happ", name, f"https://ndownloader.figshare.com/files/{fid}", "md5", checksum)
             for name, fid, checksum in HAPP_FILES]
manifest += [("uw", name, f"https://raw.githubusercontent.com/ubicomplab/oximetry-phone-cam-data/{UW_COMMIT}/data/gt/{name}",
              "git_blob", checksum) for name, checksum in UW_FILES]
if not SOURCES or not set(SOURCES) <= {"dry_apnea", "happ", "uw"}:
    raise ValueError("Select at least one supported study in SOURCES.")
manifest = [m for m in manifest if m[0] in SOURCES]
if QUICK:
    happ_names = {name for name, _, _ in HAPP_FILES[:8]}
    manifest = [m for m in manifest if m[0] != "happ" or m[1] in happ_names]

def verified(path, method, expected):
    data = path.read_bytes()
    digest = (hashlib.md5(data).hexdigest() if method == "md5" else
              hashlib.sha1(b"blob " + str(len(data)).encode() + b"\x00" + data).hexdigest())
    return digest == expected

wanted = {m[1].casefold() for m in manifest}
attached = {}
if DATA_ROOT.is_dir():
    for folder, _, names in os.walk(DATA_ROOT):
        for name in names:
            if name.casefold() in wanted:
                attached.setdefault(name.casefold(), []).append(Path(folder, name))

def source_file(item):
    source, name, url, method, digest = item
    cached = CACHE / source / name
    for path in [*attached.get(name.casefold(), []), cached]:
        if path.is_file() and verified(path, method, digest):
            return source, name, path
    if not ALLOW_DOWNLOAD:
        raise FileNotFoundError(f"Attach the original {source}/{name} below {DATA_ROOT} or enable Internet/download.")
    cached.parent.mkdir(parents=True, exist_ok=True)
    last_error = None
    partial = cached.with_suffix(cached.suffix + ".part")
    for attempt in range(3):
        try:
            resp = requests.get(url, timeout=(12, 90), headers={"User-Agent":"Oxygen-concern-research-notebook/1.0"})
            resp.raise_for_status()
            partial.write_bytes(resp.content)
            if not verified(partial, method, digest):
                raise ValueError("Checksum mismatch; check source version or attach the pinned original")
            partial.replace(cached)
            return source, name, cached
        except (requests.RequestException, OSError, ValueError) as error:
            last_error = error
            if attempt < 2:
                time.sleep(1 + attempt)
    raise RuntimeError(f"Cannot acquire {source}/{name} from {url}: {last_error}. "
                       "Enable Kaggle Internet or attach the original file.") from last_error

with ThreadPoolExecutor(max_workers=3) as pool:
    files = list(pool.map(source_file, manifest))
print(pd.Series([s for s, _, _ in files]).value_counts().rename("verified files").to_string())

In [ ]:
def make_record(source, subject, seconds, hr, spo2):
    frame = pd.DataFrame({"second": pd.to_numeric(seconds, errors="coerce"),
                          "hr": pd.to_numeric(hr, errors="coerce"),
                          "spo2": pd.to_numeric(spo2, errors="coerce")})
    frame["hr"] = frame.hr.where(np.isfinite(frame.hr) & frame.hr.gt(0))
    frame["spo2"] = frame.spo2.where(frame.spo2.gt(0) & frame.spo2.le(100))
    return {"source": source, "subject": f"{source}/{subject}", "frame": frame.reset_index(drop=True)}

def dry_records(path):
    workbook = openpyxl.load_workbook(path, read_only=True, data_only=True)
    signals = {}
    try:
        for sheet in ("FC", "SpO2"):
            rows = workbook[sheet].iter_rows(values_only=True)
            header = next(rows)
            indexes = [(i, int(pd.to_timedelta(str(h)).total_seconds())) for i, h in enumerate(header)
                       if re.fullmatch(r"\d{1,2}:\d{2}:\d{2}", str(h))]
            if [s for _, s in indexes] != list(range(360)):
                raise ValueError(f"Unexpected elapsed time columns in {sheet}")
            signals[sheet] = {int(row[0]): [row[i] for i, _ in indexes] for row in rows
                              if row[1] in {"AG", "CG"} and row[0] is not None}
    finally:
        workbook.close()
    if signals["FC"].keys() != signals["SpO2"].keys():
        raise ValueError("Unmatched dry-apnea participant IDs")
    ids = sorted(signals["FC"])
    if QUICK:
        ids = sorted(np.random.default_rng(SEED).choice(ids, size=min(24, len(ids)), replace=False))
    return [make_record("dry_apnea", pid, np.arange(360), signals["FC"][pid], signals["SpO2"][pid]) for pid in ids]

records = []
for source, name, path in files:
    if source == "dry_apnea":
        records.extend(dry_records(path))
    else:
        csv = pd.read_csv(path)
        # UW 100004 has a blank first-column header, with valid HH:MM:SS values.
        clock = csv["Time"] if "Time" in csv.columns else csv.iloc[:, 0]
        s = pd.to_timedelta(clock.astype(str).str.strip(), errors="coerce").dt.total_seconds()
        if s.notna().mean() < 0.95 or not s.dropna().between(0, 86400, inclusive="left").all():
            raise ValueError(f"Unrecognized clock in {name}; expected HH:MM:SS")
        hr_col, o2_col = ("Heart rate", "SPO2") if source == "happ" else ("Pulse 2", "SpO2 2")
        records.append(make_record(source, path.stem, s, csv[hr_col], csv[o2_col]))

# Same study-qualified participant goes to exactly one fold before windows exist.
rng = np.random.default_rng(SEED)
split_of = {}
for source in sorted(SOURCES):
    ids = np.array(sorted({r["subject"] for r in records if r["source"] == source}))
    rng.shuffle(ids)
    n_test = max(1, round(0.2 * len(ids)))
    n_val = max(1, round(0.2 * len(ids)))
    if n_test + n_val >= len(ids):
        raise ValueError(f"{source} needs at least three independent participants")
    for group, subset in (("test", ids[:n_test]), ("validation", ids[n_test:n_test+n_val]),
                          ("train", ids[n_test+n_val:])):
        split_of.update({str(pid): group for pid in subset})

record_audit = pd.DataFrame([
    {"source": r["source"], "subject": r["subject"], "split": split_of[r["subject"]],
     "rows": len(r["frame"]), "paired": int(r["frame"][["hr", "spo2"]].notna().all(axis=1).sum())}
    for r in records])
print(record_audit.groupby(["source", "split"]).agg(people=("subject","nunique"), rows=("rows","sum"),
                                                  paired=("paired","sum")).to_string())

## Declare the reference events, missingness and optional baseline

A reference event is a contiguous run of valid source oxygen at or below the configured threshold, lasting at least the required elapsed duration. Its onset is retrospectively placed at the first sample of that run; the rule must operate causally without seeing its future duration. Gaps, duplicate times and clock reversals break reference runs. No interpolation is performed. This endpoint includes stable low oxygen that Notebook A's *new relative fall* endpoint can exclude.

Optional baseline CSV columns are `subject,baseline_spo2,verified,provenance`, where subject is an exported qualified ID such as `dry_apnea/1`. `verified` must be true and provenance must state the independent annotation/protocol establishing a pre-intervention baseline. Do not fill this file with means from the first minute unless that interval was independently confirmed appropriate. A missing subject annotation disables only that person's baseline-drop branch. The annotation is per source recording in this dataset collection; a multi-session wearable study needs session-specific annotations.

Unknown oxygen produces no numerical concern value. The replay retains an existing alert latch across a gap, resets confirmation/recovery counters, and requires new valid recovery observations to clear it. Gap duration is never counted as reassuring recovery or deterioration. Source numeric validity is only a placeholder for future device-specific measurement-quality checks.

In [ ]:
baseline_by_subject = {}
if VERIFIED_BASELINE_CSV:
    baselines = pd.read_csv(VERIFIED_BASELINE_CSV)
    required = {'subject', 'baseline_spo2', 'verified', 'provenance'}
    if not required <= set(baselines):
        raise ValueError(f'Baseline annotations need columns {sorted(required)}')
    if baselines.subject.duplicated().any():
        raise ValueError('Duplicate baseline annotation for a source recording')
    checks = baselines.verified.astype(str).str.lower().isin(['true', '1', 'yes'])
    if not checks.all() or baselines.provenance.fillna('').str.strip().eq('').any():
        raise ValueError('Every supplied baseline needs verified=true and nonempty provenance')
    value = pd.to_numeric(baselines.baseline_spo2, errors='coerce')
    if not (value.gt(0) & value.le(100)).all():
        raise ValueError('Invalid baseline oxygen')
    known = {r['subject'] for r in records}
    if not set(baselines.subject) <= known:
        raise ValueError('Baseline subject IDs must match the selected, source-qualified recordings')
    baseline_by_subject = dict(zip(baselines.subject, value))

def reference_runs(t, oxygen):
    valid = np.isfinite(t) & np.isfinite(oxygen)
    contiguous = np.r_[False, (np.diff(t) == 1) & valid[1:] & valid[:-1]]
    low = valid & (oxygen <= EVENT_SPO2)
    starts = np.flatnonzero(low & ~(np.r_[False, low[:-1]] & contiguous))
    ends = np.flatnonzero(low & ~(np.r_[low[1:], False] & np.r_[contiguous[1:], False]))
    labels = np.full(len(t), -1, dtype=int)
    events = []
    for start, end in zip(starts, ends):
        if t[end] - t[start] >= EVENT_DURATION_SECONDS:
            eid = len(events)
            labels[start:end+1] = eid
            events.append({'event': eid, 'onset_row': int(start), 'end_row': int(end),
                           'onset_s': float(t[start]), 'end_s': float(t[end])})
    return valid, contiguous, labels, events

# Boundary checks prevent confusing six observations with six elapsed seconds.
assert len(reference_runs(np.arange(5.), np.full(5, EVENT_SPO2))[3]) == 0
assert len(reference_runs(np.arange(6.), np.full(6, EVENT_SPO2))[3]) == 1
assert len(reference_runs(np.array([0.,1.,2.,4.,5.,6.]), np.full(6, EVENT_SPO2))[3]) == 0

prepared, audit_rows = [], []
for record in records:
    frame = record['frame']
    t, oxygen = frame.second.to_numpy(float), frame.spo2.to_numpy(float)
    valid, contiguous, labels, events = reference_runs(t, oxygen)
    item = dict(record, split=split_of[record['subject']], t=t, oxygen=oxygen,
                valid=valid, contiguous=contiguous, labels=labels, events=events,
                baseline=baseline_by_subject.get(record['subject']))
    prepared.append(item)
    audit_rows.append({'source': item['source'], 'subject': item['subject'], 'split': item['split'],
                      'rows': len(t), 'valid_oxygen_rows': int(valid.sum()), 'reference_events': len(events),
                      'has_verified_baseline': item['baseline'] is not None,
                      'clock_discontinuities': int(np.sum(np.diff(t) != 1))})
audit = pd.DataFrame(audit_rows)
support = audit.groupby(['source', 'split']).agg(
    participants=('subject', 'nunique'), source_rows=('rows','sum'),
    valid_oxygen_rows=('valid_oxygen_rows','sum'), reference_events=('reference_events','sum'),
    people_with_events=('reference_events', lambda x: int((x > 0).sum())),
    baseline_annotations=('has_verified_baseline','sum'))
support['valid_observation_hours'] = support.valid_oxygen_rows / 3600
support['valid_fraction'] = support.valid_oxygen_rows / support.source_rows
print(support.to_string())
print('Verified baselines:', len(baseline_by_subject), '; no baseline is invented for the other recordings.')

## Fixed candidates and causal replay

Compare an immediate threshold, a fixed confirmation/recovery rule, and a small predeclared grid of concern policies. The concern policy maps a level deficit onto 0–1, optionally takes the maximum with an annotated baseline deficit, then applies an exponential filter using only observed time. A recent downward slope can shorten its configured confirmation time by half; it cannot cancel concern about low stable oxygen. Clearing requires the lower exit score and sustained valid recovery.

The optional Gaussian mapping is a normalized **monotonic CDF**, not a bell-shaped density. It uses the same endpoints as the linear map. Neither mapping is a probability. Do not tune the oxygen endpoint to make candidate scores look good. Configurations, including severity anchors, are exported as numerical experiments and are not installed on firmware.

In [ ]:
def mapping(u, kind):
    u = min(1.0, max(0.0, float(u)))
    if kind == 'linear':
        return u
    phi = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))
    return (phi(6*u - 3) - phi(-3)) / (phi(3) - phi(-3))

def prepared_features(item, smooth, shape):
    # Causal windows are cached across candidates; clock gaps and invalid rows split history.
    key = (smooth, shape)
    cache = item.setdefault('_causal_features', {})
    if key in cache:
        return cache[key]
    filtered = np.full(len(item['t']), np.nan)
    slopes = np.full(len(item['t']), np.nan)
    starts = np.flatnonzero(~item['contiguous'])
    for lo, hi in zip(starts, np.r_[starts[1:], len(filtered)]):
        if not item['valid'][lo]: continue
        s = pd.Series(item['oxygen'][lo:hi])
        filtered[lo:hi] = s.rolling(smooth, min_periods=smooth).median().to_numpy()
        if hi-lo >= 10:
            slopes[lo+9:hi] = np.correlate(s.to_numpy(), np.arange(10)-4.5, mode='valid') / 82.5
    usable = np.isfinite(filtered)
    severity = np.full(len(filtered), np.nan)
    for i in np.flatnonzero(usable):
        value = mapping((LEVEL_START-filtered[i])/(LEVEL_START-LEVEL_FULL), shape)
        if item['baseline'] is not None:
            drop = item['baseline']-filtered[i]
            value = max(value, mapping((drop-BASELINE_DROP_START)/(BASELINE_DROP_FULL-BASELINE_DROP_START), shape))
        severity[i] = value
    cache[key] = (filtered, slopes, severity, usable)
    return cache[key]

def replay(item, cfg):
    n = len(item['t'])
    filtered_values, slopes, severities, usable = prepared_features(item, cfg['smooth_s'], cfg['mapping'])
    concern, latched = np.full(n, np.nan), np.zeros(n, bool)
    alert_id = np.full(n, -1, int)
    state, active, high_since, low_since, sequence = None, False, None, None, -1
    last_usable_t = None
    for i, t in enumerate(item['t']):
        gap = not item['contiguous'][i]
        if gap or not usable[i]:
            high_since = low_since = None; last_usable_t = None
        if not usable[i]:
            latched[i] = active; alert_id[i] = sequence if active else -1
            continue
        filtered = filtered_values[i]
        dt = 0.0 if last_usable_t is None else t-last_usable_t
        last_usable_t = t
        if cfg['family'] == 'threshold':
            state = float(filtered <= EVENT_SPO2)
            high, low = state == 1, state == 0
        elif cfg['family'] == 'disabled':
            state = 0.0; high, low = False, True
        else:
            severity = severities[i]
            if state is None or cfg['tau_s'] == 0:
                state = severity
            else:
                state += (1-math.exp(-dt/cfg['tau_s'])) * (severity-state)
            high, low = state >= ENTER_SCORE, state <= EXIT_SCORE
        concern[i] = state
        confirmation = cfg['confirm_s']
        if cfg['slope_acceleration'] and slopes[i] < 0:
            confirmation /= 2
        if not active:
            low_since = None
            high_since = t if high and high_since is None else high_since if high else None
            if high_since is not None and t-high_since >= confirmation:
                active = True; sequence += 1; high_since = None
        else:
            high_since = None
            low_since = t if low and low_since is None else low_since if low else None
            if low_since is not None and t-low_since >= cfg['recover_s']:
                active = False; low_since = None
        latched[i] = active; alert_id[i] = sequence if active else -1
    return {'concern': concern, 'usable': usable, 'latched': latched,
            'warning': usable & latched, 'alert_id': alert_id}

def config(name, family='concern', smooth=1, tau=0, confirm=0, recover=0,
           shape='linear', slope=False):
    return {'name': name, 'family': family, 'smooth_s': smooth, 'tau_s': tau,
            'confirm_s': confirm, 'recover_s': recover, 'mapping': shape, 'slope_acceleration': slope}

IMMEDIATE = config('Immediate threshold', 'threshold')
FIXED = config('Threshold + 3s confirmation / 5s recovery', 'threshold', confirm=3, recover=5)
DISABLED = config('No alerts', 'disabled')
settings = itertools.product([1, 3], [0, 3] if QUICK else [0, 3, 6],
                             [0, 3] if QUICK else [0, 3, 5], [5] if QUICK else [5, 10],
                             ['linear', 'gaussian_cdf'])
CANDIDATES = [DISABLED, IMMEDIATE, FIXED]
for smooth, tau, confirm, recover, shape in settings:
    name = f'{shape} smooth={smooth} tau={tau} confirm={confirm} recover={recover}'
    CANDIDATES.append(config(name, smooth=smooth, tau=tau, confirm=confirm, recover=recover,
                             shape=shape, slope=True))

# Deterministic behavior checks, including a missing-data gap while already in alert.
test_item = {'t': np.arange(14.), 'oxygen': np.array([89.]*6+[np.nan]*3+[98.]*5), 'baseline': None}
test_item['valid'], test_item['contiguous'], _, _ = reference_runs(test_item['t'], test_item['oxygen'])
probe = replay(test_item, FIXED)
assert probe['latched'][5] and probe['latched'][8] and np.isnan(probe['concern'][8])
assert probe['latched'][-1], 'Four elapsed recovery seconds must not clear a 5-second requirement'
for shape in ['linear', 'gaussian_cdf']:
    response = np.array([mapping(u, shape) for u in np.linspace(0, 1, 101)])
    assert np.all(np.diff(response) >= 0) and abs(response[0]) < 1e-10 and abs(response[-1]-1) < 1e-10
print('Candidate policies:', len(CANDIDATES))

## Fit by validation replay, then freeze

Rules have no learned coefficients here: **fitting means selecting a fixed parameter set** from the declared grid. Training-participant replays are reported for diagnostics; validation participants select the policy; test participants are reserved for descriptive evaluation. Any redesign after reading the test needs new independent data for a new effectiveness claim.

A warning interval can credit at most one event, and an event can credit at most one warning. Detection requires an actual usable warning during the reference event. If an interval began before onset and remains active, report that separately as a pre-existing warning and record detection delay as zero. Intervals that never overlap an uncredited event are unmatched. This prevents an indefinitely latched alert from receiving credit for every later event.

Selection maximizes mean event recall across validation sources with events, then total detected events, then shorter median detection delay, subject to the declared unmatched-start budget and **non-event warning-time** budget. Budgets are pooled; individual source rates can exceed them and must be inspected. Unlike future-prediction warning time, current warnings during a true sustained low episode are expected; total warning time is reported but not capped. The no-alert option is retained. These are study-recording denominators, not false drowning alerts per swim-hour.

In [ ]:
def outcome(item, prediction):
    warning = prediction['warning']
    usable = prediction['usable']
    ids = prediction['alert_id']
    observed_alerts = np.unique(ids[warning])
    first = {int(a): int(np.flatnonzero(warning & (ids == a))[0]) for a in observed_alerts}
    credited, event_rows = set(), []
    for event in item['events']:
        lo, hi = event['onset_row'], event['end_row']
        eligible = [int(a) for a in np.unique(ids[lo:hi+1][warning[lo:hi+1]]) if int(a) not in credited]
        if eligible:
            a = min(eligible, key=lambda a: first[a]); credited.add(a)
            inside = np.flatnonzero(warning[lo:hi+1] & (ids[lo:hi+1] == a))[0]+lo
            delay = float(item['t'][inside]-event['onset_s'])
            began_before = first[a] < lo
        else:
            a, delay, began_before = -1, np.nan, False
        event_rows.append({'source': item['source'], 'subject': item['subject'], 'split': item['split'],
                           **event, 'detected': bool(eligible), 'delay_s': delay,
                           'pre_existing_warning': began_before, 'matched_alert_id': a,
                           'any_usable_input': bool(usable[lo:hi+1].any())})
    no_event = item['valid'] & (item['labels'] < 0)
    return {'source': item['source'], 'subject': item['subject'], 'split': item['split'],
            'source_rows': len(item['t']), 'valid_rows': int(item['valid'].sum()), 'usable_rows': int(usable.sum()),
            'events': len(item['events']), 'detected': len(credited),
            'unavailable_events': sum(not e['any_usable_input'] for e in event_rows),
            'warning_rows': int(warning.sum()), 'non_event_rows': int(no_event.sum()),
            'non_event_warning_rows': int((warning & no_event).sum()),
            'alert_starts': len(observed_alerts), 'unmatched_starts': len(observed_alerts)-len(credited)}, event_rows

def aggregate(rows, events):
    total = rows.select_dtypes(include='number').sum()
    n = float(total.get('events',0)); valid = float(total.get('valid_rows',0))
    per_source = rows.groupby('source')[['events','detected']].sum()
    recall = per_source.loc[per_source.events.gt(0), 'detected'] / per_source.loc[per_source.events.gt(0), 'events']
    delays = events.loc[events.detected, 'delay_s'] if not events.empty else pd.Series(dtype=float)
    return {'people': rows.subject.nunique(), 'people_with_events': rows.loc[rows.events.gt(0),'subject'].nunique(),
            'events': int(n), 'detected': int(total.get('detected',0)),
            'event_recall': total.get('detected',0)/n if n else np.nan,
            'macro_source_recall': float(recall.mean()) if len(recall) else np.nan,
            'valid_observation_hours': valid/3600,
            'unmatched_starts_per_hour': total.get('unmatched_starts',0)/(valid/3600) if valid else np.nan,
            'non_event_warning_fraction': total.get('non_event_warning_rows',0)/total['non_event_rows'] if total.get('non_event_rows',0) else np.nan,
            'warning_fraction': total.get('warning_rows',0)/valid if valid else np.nan,
            'coverage': total.get('usable_rows',0)/valid if valid else np.nan,
            'unavailable_events': int(total.get('unavailable_events',0)),
            'median_delay_s': float(delays.median()) if len(delays) else np.nan,
            'max_delay_s': float(delays.max()) if len(delays) else np.nan}

per_record, per_event = [], []
started = time.monotonic()
for index, cfg in enumerate(CANDIDATES):
    for item in prepared:
        result, event_rows = outcome(item, replay(item, cfg))
        per_record.append({'policy': cfg['name'], **result})
        per_event.extend({'policy': cfg['name'], **e} for e in event_rows)
    if index % 8 == 0 or index == len(CANDIDATES)-1:
        print(f'Replayed {index+1}/{len(CANDIDATES)} policies ({time.monotonic()-started:.0f}s)', flush=True)
record_results = pd.DataFrame(per_record)
event_columns = ['policy','source','subject','split','event','onset_row','end_row','onset_s','end_s',
                 'detected','delay_s','pre_existing_warning','matched_alert_id','any_usable_input']
event_results = pd.DataFrame(per_event, columns=event_columns)

def policy_table(split, sources=None):
    rows = []
    for cfg in CANDIDATES:
        r = record_results[(record_results.policy == cfg['name']) & (record_results.split == split)]
        e = event_results[(event_results.policy == cfg['name']) & (event_results.split == split)]
        if sources is not None:
            r, e = r[r.source.isin(sources)], e[e.source.isin(sources)]
        if not r.empty:
            rows.append({'policy': cfg['name'], **aggregate(r, e)})
    return pd.DataFrame(rows)

def select_policy(table):
    allowed = table[(table.unmatched_starts_per_hour <= MAX_UNMATCHED_PER_HOUR) &
                    (table.non_event_warning_fraction.fillna(0) <= MAX_NON_EVENT_WARNING_FRACTION) &
                    table.detected.gt(0)].copy()
    if allowed.empty:
        return DISABLED
    allowed = allowed.sort_values(['macro_source_recall','detected','median_delay_s',
                                   'unmatched_starts_per_hour','non_event_warning_fraction','policy'],
                                  ascending=[False,False,True,True,True,True])
    return next(c for c in CANDIDATES if c['name'] == allowed.iloc[0].policy)

train_table, validation_table = policy_table('train'), policy_table('validation')
selected = select_policy(validation_table)
print('Frozen selected policy:', selected)
print(validation_table[validation_table.policy.isin([IMMEDIATE['name'], FIXED['name'], selected['name']])].to_string(index=False))

## Held-out benchmarks and transfer

Inspect all event counts and event-bearing people. A large row count does not mean many independent episodes. Compare immediate threshold, fixed persistence, selected policy and no alerts. Transfer selects the configuration using validation people from the other sources, then evaluates **all people** in the excluded source; there are no trainable rule coefficients. It is a separate source-transfer experiment, not extra independent test data to pool into the primary result.

No AUROC, precision–recall or probability calibration curve is claimed for the concern index: it is a designed severity scale. Notebook A supplies those classifier diagnostics. Here parameter tradeoffs, timing, replays, monotonic response and coverage are the important figures.

In [ ]:
test_table = policy_table('test')
benchmark_names = list(dict.fromkeys([IMMEDIATE['name'], FIXED['name'], selected['name'], DISABLED['name']]))
benchmarks = test_table[test_table.policy.isin(benchmark_names)].copy()
by_source, transfer_rows = [], []
for source in sorted(SOURCES):
    for name in benchmark_names:
        r = record_results[(record_results.policy == name) & (record_results.split == 'test') & (record_results.source == source)]
        e = event_results[(event_results.policy == name) & (event_results.split == 'test') & (event_results.source == source)]
        if not r.empty:
            by_source.append({'source': source, 'policy': name, **aggregate(r, e)})
    if SOURCE_HOLDOUT and len(SOURCES) > 1:
        others = [s for s in SOURCES if s != source]
        picked = select_policy(policy_table('validation', others))
        r = record_results[(record_results.policy == picked['name']) & (record_results.source == source)]
        e = event_results[(event_results.policy == picked['name']) & (event_results.source == source)]
        transfer_rows.append({'held_out_source': source, 'selected_policy': picked['name'], **aggregate(r,e)})
by_source = pd.DataFrame(by_source); transfer = pd.DataFrame(transfer_rows)
print('Held-out people; settings selected before this report:')
print(benchmarks.round(4).to_string(index=False))
if not transfer.empty: print('\nHeld-out source experiment:\n'+transfer.round(4).to_string(index=False))

# Keep every selected-policy test observation and every event, including misses.
replays, prediction_rows = {}, []
for item in prepared:
    if item['split'] != 'test': continue
    p = replay(item, selected); replays[item['subject']] = (item,p)
    prediction_rows.append(pd.DataFrame({'source': item['source'], 'subject': item['subject'],
        'row': np.arange(len(item['t'])), 'timestamp_s': item['t'], 'spo2': item['oxygen'],
        'reference_event': item['labels'], 'source_numeric_valid': item['valid'],
        'usable': p['usable'], 'concern_0_100': 100*p['concern'], 'latched_alert': p['latched'],
        'observed_warning': p['warning'], 'alert_id': p['alert_id']}))
predictions = pd.concat(prediction_rows, ignore_index=True)
selected_events = event_results[(event_results.policy == selected['name']) & (event_results.split == 'test')].copy()

## Evaluation figures

Save figures as PNGs and display them inline. The validation operating scatter contains declared parameter candidates rather than a probability-threshold sweep. Heatmaps show the best validation recall across the remaining declared parameters and are diagnostics, not an additional selection step. Replays cover **every test event**, plus the first unmatched alert per source when available; each event plot also shows the immediate-threshold timing baseline. No successful case selection is used.

In [ ]:
def save_figure(fig, name):
    fig.tight_layout(); fig.savefig(FIGURES / f'{name}.png', dpi=150, bbox_inches='tight')
    display(fig); plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
support[['participants','people_with_events']].plot.bar(ax=axes[0], rot=65)
support[['reference_events']].plot.bar(ax=axes[1], rot=65, legend=False)
support[['valid_observation_hours']].plot.bar(ax=axes[2], rot=65, legend=False)
for ax, title in zip(axes, ['Participant support', 'Independent reference episodes', 'Valid observation hours']): ax.set_title(title)
save_figure(fig, '01_data_support')

fig, ax = plt.subplots(figsize=(8,5))
sc = ax.scatter(validation_table.unmatched_starts_per_hour, validation_table.event_recall,
                c=validation_table.non_event_warning_fraction, cmap='viridis', s=38)
chosen = validation_table[validation_table.policy == selected['name']].iloc[0]
ax.scatter([chosen.unmatched_starts_per_hour], [chosen.event_recall], marker='*', s=220, color='red', label='Selected on validation')
ax.axvline(MAX_UNMATCHED_PER_HOUR, linestyle='--', color='grey')
ax.set(xlabel='Unmatched alert starts / valid observation hour', ylabel='Reference episodes detected',
       title='Validation parameter operating points', ylim=(-.03,1.03))
fig.colorbar(sc, ax=ax, label='Fraction of non-event time in warning'); ax.legend()
save_figure(fig, '02_validation_operating_points')

metadata = pd.DataFrame(CANDIDATES).rename(columns={'name':'policy'})
grid = validation_table.merge(metadata, on='policy'); grid = grid[grid.family == 'concern']
fig, axes = plt.subplots(1,2,figsize=(10,4))
for ax, shape in zip(axes, ['linear','gaussian_cdf']):
    pivot = grid[grid.mapping == shape].pivot_table(index='confirm_s',columns='smooth_s',values='event_recall',aggfunc='max')
    if pivot.empty: ax.text(.5,.5,'No events / undefined recall',ha='center'); continue
    im=ax.imshow(pivot.to_numpy(),vmin=0,vmax=1,aspect='auto',cmap='viridis')
    ax.set_xticks(range(len(pivot.columns)),pivot.columns); ax.set_yticks(range(len(pivot.index)),pivot.index)
    ax.set(xlabel='Smoothing seconds',ylabel='Confirmation seconds',title=shape+' · best validation recall')
    fig.colorbar(im,ax=ax)
save_figure(fig, '03_parameter_heatmaps')

fig, axes = plt.subplots(1,3,figsize=(14,4))
short = {name: ('Selected' if name == selected['name'] else 'Immediate' if name == IMMEDIATE['name'] else 'Fixed' if name == FIXED['name'] else 'Off') for name in benchmark_names}
for name in benchmark_names:
    block = by_source[by_source.policy == name]
    axes[0].plot(block.source,block.event_recall,'o-',label=short[name])
    axes[1].plot(block.source,block.unmatched_starts_per_hour,'o-',label=short[name])
    axes[2].plot(block.source,block.coverage,'o-',label=short[name])
axes[0].set(ylabel='Event recall',ylim=(-.03,1.03),title='Test performance by source')
axes[1].set(ylabel='Unmatched starts / observation hour',title='Test alert burden')
axes[2].set(ylabel='Usable / numerically valid observations',ylim=(-.03,1.03),title='Processing coverage')
axes[0].legend(); save_figure(fig,'04_test_source_comparison')

fig, axes = plt.subplots(1,2,figsize=(11,4))
delays = selected_events.loc[selected_events.detected, 'delay_s'].sort_values().to_numpy(float)
if len(delays): axes[0].step(delays,np.arange(1,len(delays)+1)/len(delays),where='post')
else: axes[0].text(.5,.5,'No detected events',ha='center',transform=axes[0].transAxes)
axes[0].set(xlabel='Detection delay after reference onset (s)',ylabel='Fraction of detected events',title='Delay; misses shown separately')
counts = [int(selected_events.detected.sum()), int((~selected_events.detected.astype(bool)).sum()), int((~selected_events.any_usable_input.astype(bool)).sum())]
axes[1].bar(['Detected','Missed','No usable input*'],counts,color=['#167d8d','#ce694b','#888888'])
axes[1].set(ylabel='Reference event count',title='*Subset of missed events')
save_figure(fig,'05_detection_delay_and_misses')

fig, axes=plt.subplots(1,2,figsize=(11,4))
oxygen_axis=np.linspace(80,100,200)
for shape in ['linear','gaussian_cdf']:
    axes[0].plot(oxygen_axis,[100*mapping((LEVEL_START-x)/(LEVEL_START-LEVEL_FULL),shape) for x in oxygen_axis],label=shape)
axes[0].set(xlabel='Measured SpO₂ (%)',ylabel='Designed severity (0–100)',title='Unfiltered numerical mapping; not probability'); axes[0].legend()
toy={'t':np.arange(100.),'oxygen':np.r_[np.full(20,98.),np.linspace(98,87,15),np.full(30,87.),np.full(35,98.)], 'baseline':None}
toy['valid'],toy['contiguous'],_,_=reference_runs(toy['t'],toy['oxygen'])
for cfg in [FIXED, selected]:
    p=replay(toy,cfg); axes[1].plot(toy['t'],100*p['concern'],label=short.get(cfg['name'],cfg['name']))
axes[1].set(xlabel='Elapsed seconds',ylabel='Concern (0–100)',title='Synthetic step replay · behavior check only'); axes[1].legend()
save_figure(fig,'06_response_functions')

def plot_replay(subject, center_row, label, filename):
    item,p=replays[subject]
    lo=max(0,int(center_row)-40); hi=min(len(item['t']),int(center_row)+90)
    # Avoid connecting backwards clock jumps or gaps with invented observations.
    take=np.arange(lo,hi); x=item['t'][take]-item['t'][int(center_row)]
    oxygen=item['oxygen'][take].copy(); score=p['concern'][take].copy()
    cuts=~item['contiguous'][take]; oxygen[cuts]=np.nan; score[cuts]=np.nan
    fig,axes=plt.subplots(3,1,figsize=(11,7),sharex=True)
    axes[0].plot(x,oxygen,color='#167d8d'); axes[0].axhline(EVENT_SPO2,ls='--',color='black',label='Numerical endpoint')
    axes[0].set(ylabel='SpO₂ (%)',title=f'{subject} · {label}'); axes[0].legend()
    axes[1].plot(x,100*score,label='Selected concern'); axes[1].axhline(100*ENTER_SCORE,ls='--',color='grey')
    axes[1].set(ylabel='Concern (0–100)',ylim=(-3,103))
    immediate=replay(item,IMMEDIATE)
    axes[2].step(x,p['warning'][take].astype(int),where='post',label='Selected warning')
    axes[2].step(x,immediate['warning'][take].astype(int)*.85,where='post',alpha=.6,label='Immediate threshold')
    axes[2].fill_between(x,0,1,where=~p['usable'][take],color='grey',alpha=.25,label='Unknown / warm-up')
    axes[2].set(ylabel='Warning / validity',xlabel='Seconds relative to marked onset/start'); axes[2].legend(loc='upper right')
    for ax in axes: ax.axvline(0,color='#ce694b',ls=':')
    save_figure(fig,filename)

for j,event in enumerate(selected_events.itertuples(index=False)):
    plot_replay(event.subject,event.onset_row,'detected' if event.detected else 'MISSED',f'event_{j:03d}')
for source in SOURCES:
    found=False
    for subject,(item,p) in replays.items():
        if item['source'] != source: continue
        used=set(selected_events.loc[(selected_events.subject == subject) & selected_events.detected,'matched_alert_id'])
        unmatched=[a for a in np.unique(p['alert_id'][p['warning']]) if a not in used]
        if unmatched:
            row=np.flatnonzero(p['warning'] & (p['alert_id'] == unmatched[0]))[0]
            plot_replay(subject,row,'unmatched alert',f'unmatched_{source}'); found=True; break
    if not found: print('No selected-policy unmatched alert to plot for',source)
if not transfer.empty:
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    axes[0].bar(transfer.held_out_source,transfer.event_recall); axes[0].set(ylabel='Event recall',ylim=(0,1.03))
    axes[1].bar(transfer.held_out_source,transfer.unmatched_starts_per_hour); axes[1].set(ylabel='Unmatched starts / observation hour')
    fig.suptitle('Leave-one-source-out rule selection'); save_figure(fig,'07_source_transfer')

## Export and interpret

Download the configuration, predictions, event matches, audit tables and figure folder together. A high index indicates the configured numerical concern rule, **not blackout probability**. This notebook cannot validate its own oxygen inputs, a target-domain false-alert rate, or detection of drowning. The default has no verified session-baseline branch. Before use on the chest wearable, independently validate SpO₂ accuracy and missingness, establish clinically justified alert boundaries, and evaluate the frozen algorithm on representative recordings.

The configuration is deliberately tagged non-deployable. Exported `replay_history` requires the **entire session history**, not a rolling 30-second window: otherwise the latch and persistence state would restart. An embedded implementation must preserve state between updates. A future predictor must be evaluated as an addition to the current detector, with compatible event definitions and real simultaneous inputs; do not average the two notebooks' scores or directly compare their different endpoint recalls. HR, depth and IMU are not trained or fabricated by this notebook.

In [ ]:
configuration = {
    'purpose':'numerical source-SpO2 event-response benchmark', 'deployable':False,
    'blackout_probability':False, 'quick_smoke_test':QUICK,
    'reference_event':{'spo2_at_or_below':EVENT_SPO2,'duration_elapsed_s':EVENT_DURATION_SECONDS},
    'severity_anchors':{'start':LEVEL_START,'full':LEVEL_FULL,'status':'unvalidated numerical examples'},
    'entry_score':ENTER_SCORE,'exit_score':EXIT_SCORE,
    'baseline_mode':'verified_annotations_only','annotated_people':len(baseline_by_subject),
    'baseline_drop_start_pp':BASELINE_DROP_START,'baseline_drop_full_pp':BASELINE_DROP_FULL,
    'selection_budgets':{'unmatched_starts_per_valid_observation_hour':MAX_UNMATCHED_PER_HOUR,
                         'non_event_warning_fraction':MAX_NON_EVENT_WARNING_FRACTION},
    'selected_policy':selected,'all_candidates':CANDIDATES,'seed':SEED,
    'clock':'nominal 1Hz; actual elapsed durations; gaps never integrated',
    'versions':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,
                'openpyxl':openpyxl.__version__,'requests':requests.__version__},
    'limitations':['same-stream numeric reference','no aquatic blackout labels','other sensing sites/devices',
                   'source-side imputation or smoothing','no verified MAX30102 chest calibration']}
(OUT/'concern_configuration.json').write_text(json.dumps(configuration,indent=2,allow_nan=False),encoding='utf-8')
pd.DataFrame(manifest,columns=['source','filename','url','checksum_type','checksum']).to_csv(OUT/'source_manifest.csv',index=False)
audit.to_csv(OUT/'participant_audit.csv',index=False); support.to_csv(OUT/'data_support.csv')
train_table.to_csv(OUT/'training_grid.csv',index=False); validation_table.to_csv(OUT/'validation_grid.csv',index=False)
benchmarks.to_csv(OUT/'test_benchmarks.csv',index=False); by_source.to_csv(OUT/'test_by_source.csv',index=False)
transfer.to_csv(OUT/'source_transfer.csv',index=False)
predictions.to_csv(OUT/'test_predictions.csv',index=False); selected_events.to_csv(OUT/'test_event_matches.csv',index=False)
record_results.to_csv(OUT/'all_policy_record_metrics.csv',index=False)
# Export the exact replay functions, constants and a history wrapper; no inspect/Jupyter magic needed.
implementation = "def mapping(u, kind):\n    u = min(1.0, max(0.0, float(u)))\n    if kind == 'linear':\n        return u\n    phi = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))\n    return (phi(6*u - 3) - phi(-3)) / (phi(3) - phi(-3))\n\ndef prepared_features(item, smooth, shape):\n    # Causal windows are cached across candidates; clock gaps and invalid rows split history.\n    key = (smooth, shape)\n    cache = item.setdefault('_causal_features', {})\n    if key in cache:\n        return cache[key]\n    filtered = np.full(len(item['t']), np.nan)\n    slopes = np.full(len(item['t']), np.nan)\n    starts = np.flatnonzero(~item['contiguous'])\n    for lo, hi in zip(starts, np.r_[starts[1:], len(filtered)]):\n        if not item['valid'][lo]: continue\n        s = pd.Series(item['oxygen'][lo:hi])\n        filtered[lo:hi] = s.rolling(smooth, min_periods=smooth).median().to_numpy()\n        if hi-lo >= 10:\n            slopes[lo+9:hi] = np.correlate(s.to_numpy(), np.arange(10)-4.5, mode='valid') / 82.5\n    usable = np.isfinite(filtered)\n    severity = np.full(len(filtered), np.nan)\n    for i in np.flatnonzero(usable):\n        value = mapping((LEVEL_START-filtered[i])/(LEVEL_START-LEVEL_FULL), shape)\n        if item['baseline'] is not None:\n            drop = item['baseline']-filtered[i]\n            value = max(value, mapping((drop-BASELINE_DROP_START)/(BASELINE_DROP_FULL-BASELINE_DROP_START), shape))\n        severity[i] = value\n    cache[key] = (filtered, slopes, severity, usable)\n    return cache[key]\n\ndef replay(item, cfg):\n    n = len(item['t'])\n    filtered_values, slopes, severities, usable = prepared_features(item, cfg['smooth_s'], cfg['mapping'])\n    concern, latched = np.full(n, np.nan), np.zeros(n, bool)\n    alert_id = np.full(n, -1, int)\n    state, active, high_since, low_since, sequence = None, False, None, None, -1\n    last_usable_t = None\n    for i, t in enumerate(item['t']):\n        gap = not item['contiguous'][i]\n        if gap or not usable[i]:\n            high_since = low_since = None; last_usable_t = None\n        if not usable[i]:\n            latched[i] = active; alert_id[i] = sequence if active else -1\n            continue\n        filtered = filtered_values[i]\n        dt = 0.0 if last_usable_t is None else t-last_usable_t\n        last_usable_t = t\n        if cfg['family'] == 'threshold':\n            state = float(filtered <= EVENT_SPO2)\n            high, low = state == 1, state == 0\n        elif cfg['family'] == 'disabled':\n            state = 0.0; high, low = False, True\n        else:\n            severity = severities[i]\n            if state is None or cfg['tau_s'] == 0:\n                state = severity\n            else:\n                state += (1-math.exp(-dt/cfg['tau_s'])) * (severity-state)\n            high, low = state >= ENTER_SCORE, state <= EXIT_SCORE\n        concern[i] = state\n        confirmation = cfg['confirm_s']\n        if cfg['slope_acceleration'] and slopes[i] < 0:\n            confirmation /= 2\n        if not active:\n            low_since = None\n            high_since = t if high and high_since is None else high_since if high else None\n            if high_since is not None and t-high_since >= confirmation:\n                active = True; sequence += 1; high_since = None\n        else:\n            high_since = None\n            low_since = t if low and low_since is None else low_since if low else None\n            if low_since is not None and t-low_since >= cfg['recover_s']:\n                active = False; low_since = None\n        latched[i] = active; alert_id[i] = sequence if active else -1\n    return {'concern': concern, 'usable': usable, 'latched': latched,\n            'warning': usable & latched, 'alert_id': alert_id}\n"
header = '"""Frozen numerical benchmark replay; not a validated wearable controller."""\nimport math\nimport numpy as np\nimport pandas as pd\n\n'
for key in ['EVENT_SPO2','LEVEL_START','LEVEL_FULL','ENTER_SCORE','EXIT_SCORE','BASELINE_DROP_START','BASELINE_DROP_FULL']:
    header += f'{key} = {globals()[key]!r}\n'
header += f'SELECTED_POLICY = {selected!r}\n\n'
wrapper = """
def replay_history(timestamps, spo2, baseline=None):
    t = np.asarray(timestamps, dtype=float)
    oxygen = np.asarray(spo2, dtype=float)
    if t.ndim != 1 or oxygen.shape != t.shape or len(t) == 0:
        raise ValueError("Supply equally sized, nonempty 1-D timestamp and oxygen histories")
    if baseline is not None and not (0 < float(baseline) <= 100):
        raise ValueError("Baseline must be independently verified and in (0,100]")
    valid = np.isfinite(t) & np.isfinite(oxygen) & (oxygen > 0) & (oxygen <= 100)
    contiguous = np.r_[False, (np.diff(t) == 1) & valid[1:] & valid[:-1]]
    item = {"t":t, "oxygen":oxygen, "valid":valid, "contiguous":contiguous, "baseline":baseline}
    return replay(item, SELECTED_POLICY)
"""
(OUT/'replay_reference.py').write_text(header+implementation+'\n'+wrapper,encoding='utf-8')
loaded=json.loads((OUT/'concern_configuration.json').read_text(encoding='utf-8'))
assert loaded['deployable'] is False and loaded['selected_policy'] == selected
import importlib.util
spec=importlib.util.spec_from_file_location('exported_concern_replay',OUT/'replay_reference.py')
exported=importlib.util.module_from_spec(spec); spec.loader.exec_module(exported)
check_item, expected = next(iter(replays.values()))
actual=exported.replay_history(check_item['t'],check_item['oxygen'],check_item['baseline'])
assert np.allclose(actual['concern'],expected['concern'],equal_nan=True)
assert np.array_equal(actual['latched'],expected['latched'])
print('Saved research configuration; no firmware was changed.')
print('\n'.join(str(p.relative_to(OUT)) for p in sorted(OUT.rglob('*')) if p.is_file()))